In [1]:
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, roc_auc_score
import pandas as pd
from openai import OpenAI
import time

In [2]:
data = pd.read_csv('data/disaster_tweets/train.csv')

In [3]:
data.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [4]:
data = data.sample(500, random_state=12).reset_index(drop=True)

In [5]:
data.shape

(500, 5)

In [6]:
client = OpenAI(api_key="")

In [7]:
strategy = "few_shot"

In [8]:
prompt_template_zero_shot = """
Instructions:

You have to analyze the following tweet and to determine if it speaks about a real desaster or not. Answer with "1" if the tweet speaks about a real disaster and with "0" if not. Don't add any other information in your answer.

--------------------------
Tweet:

{text}
--------------------------
Your answer (only a "1" or a "0"):
"""

In [9]:
prompt_template_few_shot = """
Instructions:
Your task is to analyze the following tweet and determine if it is talking about a real disaster. A real disaster can include, but is not limited to, events such as earthquakes, hurricanes, fires, floods, major accidents, etc. If the tweet refers to a real disaster, respond with 1. If not, respond with 0.

Your response should only be the number 1 or 0.

Considerations:
Real Disasters: Significant events that impact people, property, or the environment.
Not Disasters: Personal opinions, jokes, fake news, or events that do not qualify as a disaster.

Examples:
Tweet: "A 7.5 magnitude earthquake has shaken the city, causing significant damage and injuries."
Expected Response: 1

Tweet: "I'm so tired that my house looks like a disaster after last night's party!"
Expected Response: 0

Tweet: "Uncontrolled wildfire in the north of the country. Evacuate immediately."
Expected Response: 1

Tweet: "It rained a lot yesterday, but today is sunny and beautiful."
Expected Response: 0

Tweet to Analyze:
Tweet: "{text}"

Response:
Result (1 or 0):
"""

In [10]:
prompt_template = prompt_template_zero_shot if strategy == "zero_shot" else prompt_template_few_shot

In [11]:
start = time.time()

In [12]:
predictions = []
for index, row in data.iterrows():
    prompt = prompt_template.format(text=row['text'])
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    answer = response.choices[0].message.content
    try:
        predictions.append(int(answer[0]))
    except:
        print("-------------------------------------")
        print(f"{index}: {answer}")
    if index % 50 == 0:
        print(index)

0
50
100
150
200
250
-------------------------------------
293: 
0
300
350
400
450


In [13]:
print(f"--- {time.time() - start} seconds --- 500 tweets")

--- 290.7089195251465 seconds --- 500 tweets


In [16]:
data["predictions"] = predictions

In [17]:
data.to_csv("./data/disaster_tweets/predictions_gpt35.csv")

In [18]:
accuracy_score(data["target"], data["predictions"])

0.776

In [19]:
recall_score(data["target"], data["predictions"])

0.48058252427184467

In [20]:
precision_score(data["target"], data["predictions"])

0.9519230769230769

In [21]:
f1_score(data["target"], data["predictions"])

0.6387096774193549

In [22]:
roc_auc_score(data["target"], data["predictions"])

0.731787860775378

------ GPT 3.5 Turbo Model ZeroShot ------

CPU time: 297.55 seconds

Accuracy: 0.784

Recall: 0.519

Precision: 0.922

F1: 0.665

AUC: 0.744

------ GPT 3.5 Turbo Model FewShot ------

CPU time: 290.71 seconds

Accuracy: 0.776

Recall: 0.481

Precision: 0.952

F1: 0.639

AUC: 0.732